![Project](https://img.shields.io/badge/NS%20Health%20%26%20Population%20Analytics-1B3A6B?style=for-the-badge&logoColor=white)

# 🏥 Chronic Disease Prevalence — Data Cleaning Pipeline
![Python](https://img.shields.io/badge/Python_3.12-C9A020?style=flat-square&logo=python&logoColor=white) ![pandas](https://img.shields.io/badge/pandas-1A6B5A?style=flat-square&logo=pandas&logoColor=white) ![Source](https://img.shields.io/badge/Source-NS_Health_Authority-1A6B5A?style=flat-square)

> **Analytical Question:** Which NS health zones show the highest rates of chronic disease (diabetes, cardiovascular disease, COPD, asthma, hypertension)?

This notebook cleans five chronic disease prevalence datasets from the NS Health Authority and stacks them into a single unified dataset, enabling cross-disease and cross-zone comparison in Power BI.

---

## 📦 Source Data

| File | Disease | Gender Breakdown | Age Groups |
|------|---------|-----------------|------------|
| `Diabetes Crude Prevalence.csv` | Diabetes | F / M | 10-year bands |
| `Hypertension Crude Prevalence.csv` | Hypertension | F / M | 10-year bands |
| `Ischemic Heart Disease Crude Prevalence.csv` | IHD | F / M | 10-year bands |
| `Acute Myocardial Infarction (AMI) Crude Prevalence.csv` | AMI | All (no sex split) | 20-year bands |
| `Asthma Crude Prevalence.csv` | Asthma | F / M | 10-year bands |

**Source:** NS Health Authority · CCDSS (Canadian Chronic Disease Surveillance System) · 2000–2022

---

## 🗺️ NS Health Zones

| Code | Name | Region |
|------|------|--------|
| Zone 1 | Western | Annapolis Valley, South Shore, Yarmouth |
| Zone 2 | Northern | Truro, Pictou, Cumberland |
| Zone 3 | Eastern | Cape Breton, Antigonish |
| Zone 4 | Central | Halifax Regional Municipality (HRM) |

---

## 🔧 Cleaning Steps Overview

| Step | Action |
|------|--------|
| 1 | Imports |
| 2 | Zone splitter helper (`'Zone 4 - Central'` → code + name) |
| 3 | Generic cleaner — normalizes all 5 files to a common schema |
| 4 | AMI special handling — force `Gender = 'All'` (no sex breakdown in source) |
| 5 | Stack all 5 diseases into one DataFrame |
| 6 | Validate & export |

---


## Step 1 — Imports


In [ ]:
import pandas as pd


## Step 2 — Zone Splitter Helper

The raw zone column looks like `'Zone 4 - Central'`. We split it into two separate columns:
- `Health Zone` → `'Zone 4'` (used as the join key to `DimHealthZone`)
- `Health Zone Name` → `'Central'` (used as the display label in Power BI)


In [ ]:
def split_zone(zone_str):
    """
    Splits 'Zone 4 - Central' into ('Zone 4', 'Central').

    Used to normalize the zone column into separate code and name fields,
    matching the DimHealthZone schema in the star schema.
    """
    zone_code, zone_name = zone_str.split('-', 1)
    return zone_code.strip(), zone_name.strip()


## Step 3 — Generic Chronic Disease Cleaner

A single reusable function handles all 5 disease files. Key operations:

- **Normalize column names** — StatCan/NSHA files use inconsistent casing; we lowercase everything first
- **Standardize column names** — map source column names (`sex`, `age_group`, `year`) to our schema
- **Parse dates** — convert year column to `datetime64` for Power BI time intelligence
- **Gender handling** — pass `force_gender='All'` for AMI (which has no sex breakdown in source)
- **Zone split** — apply `split_zone()` to create `Health Zone` and `Health Zone Name`
- **Add disease label** — each file represents one disease; we tag it explicitly for the stacked output


In [ ]:
def clean_chronic_file(file_path, disease_name, prevalence_column, force_gender=None):
    """
    Generic cleaner for NS Health Authority chronic disease CSVs.

    Parameters
    ----------
    file_path        : path to the source CSV
    disease_name     : label to add e.g. 'Diabetes'
    prevalence_column: name of the prevalence count column in source
    force_gender     : if set, overrides the gender column (used for AMI)

    Returns
    -------
    DataFrame with standardized schema:
      Date | agegroup | Gender | Health Zone | Health Zone Name |
      Chronic Disease | prevalence | population
    """
    df = pd.read_csv(file_path)

    # ── Normalize column names to lowercase ─────────────────
    df.columns = df.columns.str.lower().str.strip()

    # ── Standardize column names to our schema ──────────────
    df = df.rename(columns={
        'sex'             : 'gender',
        'age_group'       : 'agegroup',
        'year'            : 'date',
        prevalence_column : 'prevalence'    # varies per disease file
    })

    # ── Parse date to datetime ───────────────────────────────
    df['date'] = pd.to_datetime(df['date'])

    # ── Gender handling ──────────────────────────────────────
    if force_gender is not None:
        # AMI has no sex breakdown — tag all rows as 'All'
        df['gender'] = force_gender
    else:
        df['gender'] = df['gender'].replace({'M': 'M', 'F': 'F', 'All': 'All'})

    # ── Split zone into code + name ──────────────────────────
    df[['Health Zone', 'Health Zone Name']] = (
        df['zone'].apply(lambda x: pd.Series(split_zone(x)))
    )

    # ── Add disease label ────────────────────────────────────
    df['Chronic Disease'] = disease_name

    # ── Select final schema columns ──────────────────────────
    df = df[[
        'date', 'agegroup', 'gender',
        'Health Zone', 'Health Zone Name',
        'Chronic Disease', 'prevalence', 'population'
    ]]

    # ── Final column rename for consistency ─────────────────
    df = df.rename(columns={'date': 'Date', 'gender': 'Gender'})

    return df


## Step 4 — Process All 5 Disease Files

Each disease file is passed through the generic cleaner.

**AMI note:** The AMI source file has no sex breakdown — all records represent both sexes combined. We use `force_gender='All'` to tag these rows correctly rather than leaving the column blank or inheriting a wrong value.


In [ ]:
# ── Standard diseases (have F/M gender breakdown) ──────────
diabetes_df = clean_chronic_file(
    './data/Diabetes Crude Prevalence.csv',
    'Diabetes',
    'prevalence'
)

hypertension_df = clean_chronic_file(
    './data/Hypertension Crude Prevalence.csv',
    'Hypertension',
    'hypertension_count'       # NOTE: different column name in source
)

ihd_df = clean_chronic_file(
    './data/Ischemic Heart Disease Crude Prevalence.csv',
    'Ischemic Heart Disease',
    'prevalence'
)

asthma_df = clean_chronic_file(
    './data/Asthma Crude Prevalence.csv',
    'Asthma',
    'prevalence'
)

# ── AMI: no sex breakdown → force Gender = 'All' ───────────
ami_df = clean_chronic_file(
    './data/Acute Myocardial Infarction (AMI) Crude Prevalence.csv',
    'Acute Myocardial Infarction (AMI)',
    'prevalence',
    force_gender='All'
)

print("Files processed:")
for name, df in [
    ('Diabetes', diabetes_df), ('Hypertension', hypertension_df),
    ('IHD', ihd_df), ('Asthma', asthma_df), ('AMI', ami_df)
]:
    print(f"  {name:30s}: {df.shape[0]:5d} rows | Gender values: {df.Gender.unique().tolist()}")


## Step 5 — Stack Into One Dataset

`pd.concat()` stacks all 5 disease DataFrames vertically (equivalent to SQL `UNION ALL`).
The result is one unified table where `Chronic Disease` is the filter column in Power BI.


In [ ]:
chronic_diseases = pd.concat(
    [diabetes_df, hypertension_df, ihd_df, ami_df, asthma_df],
    ignore_index=True
)

# ── Validate: each disease should have same row count ───────
print("Row counts by disease (expect equal for F/M diseases):")
print(chronic_diseases['Chronic Disease'].value_counts())
print()

import os
os.makedirs("./clean", exist_ok=True)
chronic_diseases.to_csv('./clean/chronic_diseases.csv', index=False)

print("\n✅ Saved: ./clean/chronic_diseases.csv")
print(f"✅ Total rows: {len(chronic_diseases)}")
print()
print(chronic_diseases.head())


## ✅ Output Summary

| Output | Location | Rows | Diseases |
|--------|----------|------|---------|
| Clean CSV | `./clean/chronic_diseases.csv` | 6,440 | 5 |

**Final schema:** `Date` \| `agegroup` \| `Gender` \| `Health Zone` \| `Health Zone Name` \| `Chronic Disease` \| `prevalence` \| `population`

**Feeds into:**
- 📊 Power BI — Health Outcomes page (bar chart by health zone, line chart over time)
- 📊 Power BI — Demographic Deep Dive (stacked bar by age × gender × disease)
- 🧮 `Health_Outcome_Index.ipynb` — chronic disease burden component of HOI

> ⚠️ **AMI note:** AMI rows have `Gender = 'All'` — filter `Gender IN ('F','M')` when doing sex-specific comparisons to avoid double-counting.


---

*Nova Scotia Health & Population Analytics · NS Health Authority / CCDSS*
